<a href="https://colab.research.google.com/github/aabdelfattah/groot-colab/blob/main/gr00t_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Running GR00T N1 Inference — Hands-On Guide

## What is GR00T N1?

GR00T is a **Vision-Language-Action (VLA)** model — not an LLM. It doesn't generate text.

**Inputs:**
- **Camera images** (RGB video frames) — what the robot "sees"
- **Language instruction** — a natural language task like `"pick up the red cube"`
- **Robot state** (proprioception) — current joint positions, gripper state, etc.

**Output:**
- **Action trajectory** — a sequence of joint angle deltas (radians, m/s) that the robot should execute next

---

## GPU Requirements

| Model | VRAM Needed | Works on |
|-------|-------------|----------|
| GR00T-N1-2B | ~16GB | Free Colab T4 |
| GR00T-N1.6-3B | ~24GB | Colab Pro A100, RTX 4090 |

## Step 1: Install Dependencies

In [9]:
# Check latest release tag and clone Isaac GR00T repository
import subprocess

# Clone the repo first
!git clone https://github.com/NVIDIA/Isaac-GR00T.git
%cd Isaac-GR00T

# Fetch tags and checkout the latest stable release
!git fetch --tags
latest_tag = subprocess.check_output(['git', 'describe', '--tags', '--abbrev=0']).decode().strip()
print(f"Latest release tag: {latest_tag}")
!git checkout {latest_tag}

# Install flash-attn from pre-built wheel (avoids compilation issues on Colab)
!pip install ninja packaging --quiet

import torch
cuda_version = torch.version.cuda.replace(".", "")[:3]  # e.g. "124"
torch_version = torch.__version__.split("+")[0]  # e.g. "2.6.0"
python_version = f"cp{subprocess.check_output(['python', '-c', 'import sys; print(f\"{sys.version_info.major}{sys.version_info.minor}\")']).decode().strip()}"
print(f"Detected: CUDA {torch.version.cuda}, PyTorch {torch_version}, Python {python_version}")

# Use pre-built wheel from flash-attn GitHub releases (much faster than compiling)
!pip install flash-attn --no-build-isolation --quiet 2>/dev/null || \
    pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu{cuda_version}torch{torch_version}cxx11abiFALSE-{python_version}-{python_version}-linux_x86_64.whl" --quiet 2>/dev/null || \
    echo "⚠️ flash-attn wheel not found for this environment, trying compilation with limited jobs..." && \
    MAX_JOBS=2 pip install flash-attn --no-build-isolation --quiet

# Now install gr00t
!pip install -e . --quiet
!pip install huggingface_hub --quiet

print(f"\n✓ Installed Isaac-GR00T @ {latest_tag}")

Cloning into 'Isaac-GR00T'...
remote: Enumerating objects: 1749, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 1749 (delta 80), reused 43 (delta 43), pack-reused 1630 (from 3)
Receiving objects: 100% (1749/1749), 61.36 MiB | 43.90 MiB/s, done.
Resolving deltas: 100% (831/831), done.
Filtering content: 100% (33/33), 124.92 MiB | 19.43 MiB/s, done.
/content/Isaac-GR00T/Isaac-GR00T
Latest release tag: n1.5-release
Note: switching to 'n1.5-release'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config v

In [12]:
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

!pip install -e . --quiet



Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA: 12.8
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for gr00t (pyproject.toml) ... done


## Step 2: Check Your GPU

In [13]:
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Recommendation based on VRAM
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb < 20:
    print("\n→ Recommended: Use GR00T-N1-2B (smaller model)")
else:
    print("\n→ You can use GR00T-N1.6-3B (larger model)")

GPU: Tesla T4
VRAM: 15.6 GB

→ Recommended: Use GR00T-N1-2B (smaller model)


## Step 3: Download the Model from HuggingFace

In [16]:
from huggingface_hub import snapshot_download

# Choose model based on your GPU:
# - Use N1-2B for T4 (16GB)
# - Use N1.6-3B for A100 (80GB) or RTX 4090 (24GB)

USE_SMALL_MODEL = True  # Set to False if you have 24GB+ VRAM

if USE_SMALL_MODEL:
    model_path = snapshot_download("nvidia/GR00T-N1-2B")
else:
    model_path = snapshot_download("nvidia/GR00T-N1.6-3B")

print(f"Model downloaded to: {model_path}")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Model downloaded to: /root/.cache/huggingface/hub/models--nvidia--GR00T-N1-2B/snapshots/fc879581ca32f4f6d6e02cf0cc80452f6b0c3873


## Step 4: Download Demo Dataset

GR00T uses the **LeRobot dataset format**. NVIDIA provides example datasets.

In [15]:
from huggingface_hub import snapshot_download
import os

# Download a small demo subset (102 trajectories — smallest available)
dataset_path = os.path.join(os.getcwd(), "gr00t_demo_data")
snapshot_download(
    "nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim",
    repo_type="dataset",
    allow_patterns="unitree_g1.LMPnPAppleToPlateDC/**",
    local_dir=dataset_path,
)
dataset_path = os.path.join(dataset_path, "unitree_g1.LMPnPAppleToPlateDC")
print(f"Dataset at: {dataset_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
HTTP Error 429 thrown while requesting GET https://huggingface.co/api/datasets/nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim/tree/main?expand=false&recursive=true&limit=1000&cursor=ZXlKbWFXeGxYMjVoYldVaU9pSnphVzVuYkdWZmNHRnVaR0ZmWjNKcGNIQmxjaTVQY0dWdVJHOTFZbXhsUkc5dmNpOTJhV1JsYjNNdlkyaDFibXN0TURBd0wyOWljMlZ5ZG1GMGFXOXVMbWx0WVdkbGN5NXlhV2RvZEY5MmFXVjNMMlZ3YVhOdlpHVmZNREF3T0RVNExtMXdOQ0lzSW5SeVpXVmZiMmxrSWpvaU5UTm1PR1EyTW1Rek5qWmhZekE1WVRCbE9HWmlPRGRoWldGa05HTTNPVGN5TnpZelpqW

Fetching ... files: 0it [00:00, ?it/s]

Dataset at: /content/Isaac-GR00T/Isaac-GR00T/gr00t_demo_data/unitree_g1.LMPnPAppleToPlateDC


## Step 5: Load the GR00T Policy Model

In [19]:
!pip install -e . --quiet

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for gr00t (pyproject.toml) ... done


In [2]:
%cd /content/Isaac-GR00T/Isaac-GR00T
!pip install -e . --quiet

/content/Isaac-GR00T/Isaac-GR00T
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for gr00t (pyproject.toml) ... done


In [11]:
!pip install "git+https://github.com/facebookresearch/pytorch3d.git" --quiet


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [13]:
!pip install decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 84.4 MB/s eta 0:00:00


In [18]:
# Modify the gr00t/model/gr00t_n1.py file to handle the TypeError
import inspect
import os

def patch_gr00t_n1_init():
    # Derive the path to gr00t_n1.py directly, as Gr00tPolicy is not yet imported
    gr00t_n1_path = os.path.join(
        os.getcwd(), "gr00t", "model", "gr00t_n1.py"
    )

    # Check if the file exists before attempting to read/write
    if not os.path.exists(gr00t_n1_path):
        print(f"Error: gr00t_n1.py not found at {gr00t_n1_path}. Skipping patch.")
        return

    with open(gr00t_n1_path, "r") as f:
        lines = f.readlines()

    modified_lines = []
    patched = False
    for i, line in enumerate(lines):
        if "self.backbone = EagleBackbone(**config.backbone_cfg)" in line and not patched:
            indentation = " " * (len(line) - len(line.lstrip()))
            modified_lines.append(f"{indentation}backbone_cfg_filtered = config.backbone_cfg.copy()\n")
            modified_lines.append(f"{indentation}if \"allow_reshape_visual\" in backbone_cfg_filtered:\n")
            modified_lines.append(f"{indentation}    del backbone_cfg_filtered[\"allow_reshape_visual\"]\n")
            modified_lines.append(line.replace("config.backbone_cfg", "backbone_cfg_filtered"))
            patched = True
        else:
            modified_lines.append(line)

    if patched:
        with open(gr00t_n1_path, "w") as f:
            f.writelines(modified_lines)
        print(f"Patched {gr00t_n1_path} to remove 'allow_reshape_visual' from EagleBackbone config.")
    else:
        print(f"Warning: Could not find target line in {gr00t_n1_path} for patching. File might already be patched or content is different.")

patch_gr00t_n1_init()

# Now import the modules after the patch has been applied
from gr00t.model.policy import Gr00tPolicy
from gr00t.data.embodiment_tags import EmbodimentTag
from gr00t.experiment.data_config import DATA_CONFIG_MAP

# Get data config and transforms for the embodiment
data_config = DATA_CONFIG_MAP["unitree_g1"]
modality_config = data_config.modality_config()
transforms = data_config.transform()

policy = Gr00tPolicy(
    model_path=model_path,
    modality_config=modality_config,
    modality_transform=transforms,
    embodiment_tag=EmbodimentTag.NEW_EMBODIMENT,
    device="cuda",
)

print("Model loaded successfully!")

You are using a model of type gr00t_n1 to instantiate a model of type gr00t_n1_5. This is not supported for all configurations of models and can yield errors.


Model not found or avail in the huggingface hub. Loading from local path: /root/.cache/huggingface/hub/models--nvidia--GR00T-N1-2B/snapshots/fc879581ca32f4f6d6e02cf0cc80452f6b0c3873
Loading pretrained dual brain from /root/.cache/huggingface/hub/models--nvidia--GR00T-N1-2B/snapshots/fc879581ca32f4f6d6e02cf0cc80452f6b0c3873
Tune backbone vision tower: True
Tune backbone LLM: False
Tune action head projector: True
Tune action head DiT: True
Model not found or avail in the huggingface hub. Loading from local path: /root/.cache/huggingface/hub/models--nvidia--GR00T-N1-2B/snapshots/fc879581ca32f4f6d6e02cf0cc80452f6b0c3873


TypeError: EagleBackbone.__init__() got an unexpected keyword argument 'allow_reshape_visual'

## Step 6: Load a Sample Episode and Inspect the Data

In [ ]:
from gr00t.data.dataset import LeRobotSingleDataset, ModalityConfig

dataset = LeRobotSingleDataset(
    dataset_path=dataset_path,
    modality_configs=modality_config,
    embodiment_tag=EmbodimentTag.NEW_EMBODIMENT,
)

sample = dataset[0]

print("=== INPUT TO THE MODEL ===")
print(f"Keys in sample: {list(sample.keys())}")
for key, value in sample.items():
    if hasattr(value, 'shape'):
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
    elif isinstance(value, list):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {type(value).__name__}")

## Step 7: Run Inference — Get Action Predictions

In [ ]:
# The model takes the sample and predicts what the robot should do next
action = policy.get_action(sample)

print("=== OUTPUT FROM THE MODEL ===")
print(f"Action shape: {action['action'].shape}")
print(f"Action dtype: {action['action'].dtype}")
print(f"Action sample (first 5 values): {action['action'][0, :5]}")
print()
print("These are joint angle DELTAS in radians/meters.")
print("Each value tells a specific joint how much to move from its current position.")

## Step 8: Visualize Input and Predicted Actions

In [ ]:
import matplotlib.pyplot as plt

# Find the image key in the sample
image_keys = [k for k in sample.keys() if 'image' in k.lower() or 'video' in k.lower()]
print(f"Image keys found: {image_keys}")

if image_keys:
    img = sample[image_keys[0]]
    if len(img.shape) == 4:  # (T, H, W, C) - take last frame
        img = img[-1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Show the camera image (what the robot sees)
    axes[0].imshow(img)
    axes[0].set_title("Robot's View (Camera Input)")
    axes[0].axis('off')

    # Plot the predicted action trajectory
    actions = action['action']
    if len(actions.shape) == 2:
        for joint_idx in range(min(actions.shape[1], 6)):
            axes[1].plot(actions[:, joint_idx], label=f'Joint {joint_idx}')
    axes[1].set_title("Predicted Action Trajectory")
    axes[1].set_xlabel("Time Step")
    axes[1].set_ylabel("Delta (radians/meters)")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()
else:
    print("No image keys found in sample")

## Step 9: Try Different Language Instructions

See how the model generates different action trajectories based on the task instruction.

In [ ]:
import matplotlib.pyplot as plt

instructions = [
    "pick up the object",
    "move the arm to the left",
    "place the object on the table",
]

fig, axes = plt.subplots(1, len(instructions), figsize=(6*len(instructions), 4))

for i, instruction in enumerate(instructions):
    # Modify the language instruction in the sample
    lang_keys = [k for k in sample.keys() if 'lang' in k.lower() or 'task' in k.lower() or 'instruction' in k.lower()]
    modified_sample = dict(sample)
    for key in lang_keys:
        modified_sample[key] = [instruction]

    result = policy.get_action(modified_sample)
    actions = result['action']

    if len(actions.shape) == 2:
        for joint_idx in range(min(actions.shape[1], 6)):
            axes[i].plot(actions[:, joint_idx], label=f'Joint {joint_idx}')
    axes[i].set_title(f'"{instruction}"')
    axes[i].set_xlabel("Time Step")
    axes[i].legend(fontsize=7)
    axes[i].grid(True)

plt.tight_layout()
plt.show()

---

## Understanding the I/O (Interview Talking Points)

### Input (what the robot perceives):
```
┌─────────────────────────────────────────────┐
│  Camera Image(s)     →  "What do I see?"    │
│  (RGB, any resolution)                       │
│                                              │
│  Language Instruction →  "What should I do?" │
│  ("pick up the red cube")                    │
│                                              │
│  Proprioception      →  "Where am I now?"   │
│  (joint angles, gripper state)               │
└─────────────────────────────────────────────┘
```

### Output (what the robot should do):
```
┌─────────────────────────────────────────────┐
│  Action Trajectory   →  "How should I move?"│
│  (16 future timesteps of joint deltas)       │
│  Values in real physical units (rad, m/s)    │
│  State-relative (delta from current state)   │
└─────────────────────────────────────────────┘
```

### Architecture:
- **Vision encoder:** Cosmos-Reason-2B + Eagle3 ViT (processes the camera images)
- **Language encoder:** T5 transformer (encodes the task instruction)
- **Action head:** Diffusion Transformer (generates smooth action trajectories via iterative denoising)

### Key insight:
GR00T uses **flow matching** (a type of diffusion) for action generation — this is why the actions are smooth and physically plausible, unlike autoregressive token prediction. The model predicts **state-relative deltas** (not absolute positions), which makes it more robust to small errors accumulating over time.

---

## Troubleshooting

| Problem | Solution |
|---------|----------|
| CUDA OOM on T4 | Set `USE_SMALL_MODEL = True` to use `nvidia/GR00T-N1-2B` |
| `flash-attn` build fails | Already handled — we install with `--no-build-isolation` |
| `ModuleNotFoundError` | Run `!pip install -e .` again |
| Dataset download fails | Run `!huggingface-cli login` |
| Slow download | Model is ~6GB — give it a few minutes |
| No GPU available | Runtime → Change runtime type → GPU |